# Plan Phase D: NN eval 学習 モデル拡大版 (Colab T4 GPU)

**v2 (= colab_train_nn_eval_v2.ipynb) との差分**:
- **モデル拡大**: 78 → 64 → 32 (= 7,466 params) → **78 → 256 → 128 → 64** (= 62,026 params、 8.3x)
- **dropout 0.2** を各 hidden 後に追加 (= 60K params で過学習対策)
- それ以外 (= epoch 100、 cosine、 batch 1024、 early stop 15) は v2 と同じ
- 出力名は `_v3` に変更 (= v1/v2 と共存)

**期待**:
- 仮説 A (= モデル容量が限界だった): val_sign_acc 0.66 → 0.71-0.74 (+5-8 pt)
- 仮説 B (= 78 dim データの天井): 0.66 → 0.67 程度の微増 (+1 pt)

結果次第で次の打ち手 (= Phase F state encoder 化 or ensemble) を判断。

## 使い方

1. Drive `MyDrive/onepiece_nn/self_play_snapshots.jsonl` を準備 (= v1 と同じ snapshot を再利用)
2. Colab で開く → 「ランタイム > ランタイムのタイプを変更 > T4 GPU」
3. 「ランタイム > すべてのセルを実行」
4. 完了後 Drive の `nn_eval_v3.pt` + `_nn_feature_keys_v3.json` + `training_curve_v3.json` をローカル DL

T4 で約 20-40 分の見込み (= モデル拡大で per-step が 2-3x、 ただし早期停止で総 epoch は減る)。

## 1. GPU 確認

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)
assert torch.cuda.is_available(), 'GPU が有効化されていません。 ランタイム > ランタイムのタイプを変更 > T4 GPU を選択してください。'

## 2. Google Drive マウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/onepiece_nn'
SNAPSHOT_PATH = os.path.join(WORK_DIR, 'self_play_snapshots.jsonl')
OUTPUT_PT = os.path.join(WORK_DIR, 'nn_eval_v3.pt')
OUTPUT_KEYS = os.path.join(WORK_DIR, '_nn_feature_keys_v3.json')
OUTPUT_CURVE = os.path.join(WORK_DIR, 'training_curve_v3.json')

assert os.path.exists(SNAPSHOT_PATH), f'snapshot が見つからない: {SNAPSHOT_PATH}'
print('snapshot size:', os.path.getsize(SNAPSHOT_PATH) // (1024*1024), 'MB')

## 3. snapshot ロード + tensor 化

In [ ]:
import json, time
import numpy as np

N_ACTION_CATEGORIES = 9

t0 = time.time()
snapshots = []
feature_keys = None
with open(SNAPSHOT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            snap = json.loads(line)
        except json.JSONDecodeError:
            continue
        snapshots.append(snap)
        if feature_keys is None and 'features' in snap:
            feature_keys = sorted(snap['features'].keys())
print(f'loaded {len(snapshots):,} snapshots in {time.time()-t0:.1f}s, dim={len(feature_keys)}')

device = torch.device('cuda')
X_np = np.zeros((len(snapshots), len(feature_keys)), dtype=np.float32)
y_np = np.zeros((len(snapshots),), dtype=np.float32)
for i, snap in enumerate(snapshots):
    feats = snap.get('features', {})
    for j, k in enumerate(feature_keys):
        X_np[i, j] = float(feats.get(k, 0.0))
    y_np[i] = float(snap.get('winner', snap.get('final_winner', 0)))
X = torch.from_numpy(X_np).to(device)
y = torch.from_numpy(y_np).unsqueeze(1).to(device)
a = torch.zeros(len(snapshots), dtype=torch.long, device=device)

n = len(snapshots)
split = int(n * 0.8)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
a_train, a_val = a[:split], a[split:]
print(f'train: {X_train.shape[0]:,}, val: {X_val.shape[0]:,}')
del snapshots, X_np, y_np

## 4. モデル定義 (= LargerEvaluator、 78 → 256 → 128 → 64 + dropout)

In [ ]:
import torch.nn as nn

class LargerEvaluator(nn.Module):
    """Phase D: SimpleEvaluator (= 7K params) を 78 → 256 → 128 → 64 (= 62K params) に拡大。
    
    過学習対策で各 hidden 後に Dropout(0.2)。
    """
    def __init__(self, input_dim, dropout=0.2):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.value_head = nn.Linear(64, 1)
        self.policy_head = nn.Linear(64, N_ACTION_CATEGORIES)
    def forward(self, x):
        h = self.shared(x)
        return self.value_head(h), self.policy_head(h)

model = LargerEvaluator(input_dim=len(feature_keys)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'params: {n_params:,} (= v2 SimpleEvaluator 7,466 の {n_params/7466:.1f}x)')

## 5. 学習 (= v2 と同じ cosine scheduler + early stopping)

In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

EPOCHS = 100
BATCH = 1024
LR_MAX = 1e-3
LR_MIN = 1e-5
EARLY_STOP_PATIENCE = 15

train_ds = TensorDataset(X_train, y_train, a_train)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)

optimizer = optim.Adam(model.parameters(), lr=LR_MAX)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
value_criterion = nn.MSELoss()
policy_criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
epochs_since_best = 0
curve = []

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0
    n_batches = 0
    for Xb, yb, ab in train_loader:
        optimizer.zero_grad()
        v_pred, p_pred = model(Xb)
        loss_v = value_criterion(v_pred, yb)
        loss_p = policy_criterion(p_pred, ab)
        loss = loss_v + 0.5 * loss_p
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item()
        n_batches += 1

    model.eval()  # dropout を無効化
    with torch.no_grad():
        v_pred_val, _ = model(X_val)
        sign_acc = float((torch.sign(v_pred_val) == torch.sign(y_val)).float().mean().item())
    avg_loss = train_loss_sum / max(1, n_batches)
    current_lr = optimizer.param_groups[0]['lr']
    curve.append({'epoch': epoch+1, 'train_loss': avg_loss, 'val_sign_acc': sign_acc, 'lr': current_lr})
    msg = f'epoch {epoch+1:3d}/{EPOCHS}: train_loss={avg_loss:.4f}, val_sign_acc={sign_acc:.3f}, lr={current_lr:.2e}'

    if sign_acc > best_val_acc:
        best_val_acc = sign_acc
        torch.save({k: v.cpu() for k, v in model.state_dict().items()}, OUTPUT_PT)
        with open(OUTPUT_KEYS, 'w', encoding='utf-8') as f:
            json.dump(feature_keys, f, ensure_ascii=False)
        epochs_since_best = 0
        msg += f'  ✔ saved (best={sign_acc:.3f})'
    else:
        epochs_since_best += 1
    print(msg)

    scheduler.step()

    if epochs_since_best >= EARLY_STOP_PATIENCE:
        print(f'\n=== early stopping at epoch {epoch+1} (no improvement for {EARLY_STOP_PATIENCE} epochs) ===')
        break

with open(OUTPUT_CURVE, 'w', encoding='utf-8') as f:
    json.dump(curve, f, ensure_ascii=False, indent=2)
print(f'\n=== DONE. best val_sign_acc={best_val_acc:.3f} ===')
print(f'v1 baseline: 0.643')
print(f'v2 (= epoch + scheduler): 0.6587')
print(f'v3 (= モデル拡大): {best_val_acc:.4f}')
print(f'v3 vs v2 delta: {(best_val_acc - 0.6587)*100:+.1f} pt')

## 6. 成果物確認 + 学習曲線可視化

In [ ]:
import matplotlib.pyplot as plt

for p in [OUTPUT_PT, OUTPUT_KEYS, OUTPUT_CURVE]:
    if os.path.exists(p):
        print(f'  {p}: {os.path.getsize(p):,} bytes')
    else:
        print(f'  {p}: NOT FOUND')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_x = [c['epoch'] for c in curve]
ax1.plot(epochs_x, [c['train_loss'] for c in curve], label='train_loss')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(); ax1.set_yscale('log')
ax2.plot(epochs_x, [c['val_sign_acc'] for c in curve], label='val_sign_acc', color='orange')
ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='coin flip')
ax2.axhline(0.6587, color='blue', linestyle='--', alpha=0.5, label='v2 best=0.659')
ax2.axhline(best_val_acc, color='green', linestyle='--', alpha=0.5, label=f'v3 best={best_val_acc:.3f}')
ax2.set_xlabel('epoch'); ax2.set_ylabel('val sign acc'); ax2.legend()
plt.tight_layout(); plt.show()

print('\n--- 次の手順 ---')
print('1. Drive の nn_eval_v3.pt と _nn_feature_keys_v3.json と training_curve_v3.json を共有 URL で取得')
print('2. それぞれ右クリック > リンクを取得 > リンクを知っている全員 > URL コピー > Claude に貼り付け')